In [ ]:
import re
import pandas as pd


In [6]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+\.\S+/\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [7]:
def load_data(path):
    df = pd.read_csv(path, encoding="latin1")
    df.columns = ["target", "id", "date", "flag", "user", "text"]
    df = df.dropna(subset=["text", "target"])
    df = df[df["target"].isin([0, 4])]
    df["text"] = df["text"].astype(str).apply(preprocess)
    df["target"] = df["target"].map({0: 0, 4: 1}).astype(int)

    return df

In [8]:
def balanced_sample(df, n_total, label_col="target"):
    classes = sorted(df[label_col].unique())
    n_per_class = n_total // len(classes)

    parts = []
    used_indices = []

    for c in classes:
        part = df[df[label_col] == c].sample(n=n_per_class, random_state=42)
        parts.append(part)
        used_indices.extend(part.index)

    return pd.concat(parts), used_indices


In [12]:
df = load_data("training.1600000.processed.noemoticon.csv")

# TEST
test_df, test_idx = balanced_sample(df, 10_000)
remaining = df.drop(index=test_idx)

# VAL
val_df, val_idx = balanced_sample(remaining, 20_000)
remaining = remaining.drop(index=val_idx)

# TRAIN
train_df, train_idx = balanced_sample(remaining, 100_000)

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 100000
Val size:   20000
Test size:  10000


In [18]:
train_df.to_parquet("train.parquet")
val_df.to_parquet("val.parquet")
test_df.to_parquet("test.parquet")

print("Datasets saved!")

Datasets saved!
